# Тема курсового проекта
Генерация конспекта по видео лекции: На вход подается видео запись лекции, задача: распознать содержание (SpeechToText) и сгенерировать краткую выжимку с основными тезисами - конспект.

## Рассматриваемые модели

T5-small ~240 MB - лёгкая базовая модель

BART-base ~550 MB - предобучена на суммаризации

Flan-T5-base ~990 MB - улучшенная версия T5 

## обучение
Все три модели обучаются одинаково через Seq2SeqTrainer из библиотеки
HuggingFace Transformers на одном и том же датасете из Лабы 3.

## Метрика сравнения
Модели сравниваются по ROUGE (ROUGE-1, ROUGE-2, ROUGE-L) на тестовой
выборке test.csv, которая не участвовала в обучении.

# Установка библиотек

In [2]:
%pip install transformers datasets rouge-score sentencepiece accelerate pandas torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Загрузка датасета

In [3]:
import pandas as pd

# Загружаем датасет из Лабы 3
train = pd.read_csv('dataset/train.csv')
val = pd.read_csv('dataset/val.csv')
test = pd.read_csv('dataset/test.csv')

train = train[['transcript', 'summary']]
val = val[['transcript', 'summary']]
test = test[['transcript', 'summary']]

# Модель 1 - T5

In [5]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import torch

MODEL_NAME = "google-t5/t5-small"

tokenizer_t5 = AutoTokenizer.from_pretrained(MODEL_NAME)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Модель загружена: {MODEL_NAME}")
print(f"Параметров: {model_t5.num_parameters():,}")

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 128
PREFIX = "summarize: "

def tokenize(batch):
    inputs = [PREFIX + text for text in batch['transcript']]
    model_inputs = tokenizer_t5(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
    )
    labels = tokenizer_t5(
        text_target=batch['summary'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds_t5 = Dataset.from_pandas(train).map(tokenize, batched=True)
val_ds_t5 = Dataset.from_pandas(val).map(tokenize, batched=True)

training_args_t5 = Seq2SeqTrainingArguments(
    output_dir='./results/t5-small',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs/t5-small',
    logging_steps=50,
    evaluation_strategy='epoch', 
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

data_collator_t5 = DataCollatorForSeq2Seq(tokenizer_t5, model=model_t5)

trainer_t5 = Seq2SeqTrainer(
    model=model_t5,
    args=training_args_t5,
    train_dataset=train_ds_t5,
    eval_dataset=val_ds_t5,
    tokenizer=tokenizer_t5,
    data_collator=data_collator_t5,
)

print("Начинаем обучение T5-small...")
trainer_t5.train()
print("✅ Обучение T5-small завершено")

Модель загружена: google-t5/t5-small
Параметров: 60,506,624


Map: 100%|██████████| 310/310 [00:00<00:00, 452.31 examples/s]


Начинаем обучение T5-small...


e:\labs\kafedra\.venv\lib\site-packages\transformers\optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  6%|▌         | 50/813 [01:35<27:24,  2.16s/it]

{'loss': 2.7408, 'learning_rate': 2.5e-05, 'epoch': 0.18}


...

{'loss': 2.2598, 'learning_rate': 1.0518934081346423e-06, 'epoch': 2.95}


                                                 
100%|██████████| 813/813 [30:57<00:00,  2.08s/it]

{'eval_loss': 2.106924057006836, 'eval_runtime': 28.0296, 'eval_samples_per_second': 11.06, 'eval_steps_per_second': 1.391, 'epoch': 3.0}


100%|██████████| 813/813 [30:59<00:00,  2.29s/it]

{'train_runtime': 1859.3307, 'train_samples_per_second': 3.495, 'train_steps_per_second': 0.437, 'train_loss': 2.344302657198935, 'epoch': 3.0}
✅ Обучение T5-small завершено


In [6]:
import numpy as np

def generate_summary(model, tokenizer, text, prefix="summarize: ", max_input=512, max_output=128):
    input_text = prefix + text
    inputs = tokenizer(
        input_text,
        max_length=max_input,
        truncation=True,
        return_tensors='pt'
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_output,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

model_t5.eval()
model_t5.to('cuda' if torch.cuda.is_available() else 'cpu')

print(" предсказания T5-small на тестовой выборке")

predictions_t5 = []
references_t5 = []

for i, row in test.iterrows():
    pred = generate_summary(model_t5, tokenizer_t5, row['transcript'])
    predictions_t5.append(pred)
    references_t5.append(row['summary'])

    if i % 50 == 0:
        print(f"  Обработано: {i}/{len(test)}")

print("✅ Инференс завершён")

 предсказания T5-small на тестовой выборке
  Обработано: 0/620
  Обработано: 50/620


KeyboardInterrupt: 